## Import Libraries

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

import datetime as dt

import warnings
warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

%matplotlib inline

pd.set_option('display.max_columns', 250) 
plt.style.use('seaborn-v0_8-pastel') 

## Checking Dataset

In [2]:
df_mortgage = pd.read_csv("../../Data/mortgage_covenant_data.csv")

In [3]:
df_mortgage.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100931 entries, 0 to 100930
Data columns (total 14 columns):
 #   Column                             Non-Null Count   Dtype  
---  ------                             --------------   -----  
 0   activity_year                      100931 non-null  int64  
 1   census_tract                       99915 non-null   float64
 2   derived_race                       100931 non-null  object 
 3   action_taken                       100931 non-null  int64  
 4   loan_amount                        100931 non-null  float64
 5   property_value                     83491 non-null   float64
 6   income                             88200 non-null   float64
 7   interest_rate                      78125 non-null   float64
 8   tract_minority_population_percent  100931 non-null  float64
 9   tract_to_msa_income_percentage     100931 non-null  float64
 10  denial_reason-1                    100931 non-null  int64  
 11  covenant_count                     1009

In [4]:
df_mortgage

,activity_year,census_tract,derived_race,action_taken,loan_amount,property_value,income,interest_rate,tract_minority_population_percent,tract_to_msa_income_percentage,denial_reason-1,covenant_count,was_approved,covenant_density
0,2023,2.703706e+10,Race Not Available,6,185000.0,195000.0,NaN,6.125,40.37,73.48,10,0.0,False,NaN
1,2023,2.714103e+10,Race Not Available,6,375000.0,385000.0,NaN,6.625,10.64,121.26,10,0.0,False,NaN
2,2023,2.702395e+10,Race Not Available,6,105000.0,105000.0,NaN,6.125,8.92,90.13,10,0.0,False,NaN
3,2023,2.717110e+10,Race Not Available,6,285000.0,285000.0,NaN,6.750,13.27,104.37,10,0.0,False,NaN
4,2023,2.715948e+10,Race Not Available,6,175000.0,175000.0,NaN,6.875,5.87,82.22,10,0.0,False,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
100926,2023,2.700902e+10,Race Not Available,6,255000.0,265000.0,NaN,6.125,4.88,106.18,10,0.0,False,NaN
100927,2023,2.706778e+10,Race Not Available,6,185000.0,335000.0,NaN,6.125,46.58,74.68,10,0.0,False,NaN
100928,2023,2.714948e+10,Race Not Available,6,65000.0,75000.0,NaN,6.750,20.14,118.83,10,0.0,False,NaN
100929,2023,2.705913e+10,Race Not Available,6,545000.0,525000.0,NaN,5.000,6.84,90.07,10,0.0,False,NaN


## Algorithmic Bias

### Creating X and y datasets

In [5]:
#Creating dummy variables
dummies_race = pd.get_dummies(df_mortgage['derived_race'], drop_first = False, dtype='int')
df_mortgage = pd.concat([df_mortgage, dummies_race], axis = 1)

#Converting 'was_approved' into 1s and 0s for analysis
df_mortgage['was_approved'] = df_mortgage['was_approved'].apply(lambda x: 1 if str(x).strip().lower() == 'approved' else 0)

In [6]:
df_mortgage

,activity_year,census_tract,derived_race,action_taken,loan_amount,property_value,income,interest_rate,tract_minority_population_percent,tract_to_msa_income_percentage,denial_reason-1,covenant_count,was_approved,covenant_density,2 or more minority races,American Indian or Alaska Native,Asian,Black or African American,Free Form Text Only,Joint,Native Hawaiian or Other Pacific Islander,Race Not Available,White
0,2023,2.703706e+10,Race Not Available,6,185000.0,195000.0,NaN,6.125,40.37,73.48,10,0.0,0,NaN,0,0,0,0,0,0,0,1,0
1,2023,2.714103e+10,Race Not Available,6,375000.0,385000.0,NaN,6.625,10.64,121.26,10,0.0,0,NaN,0,0,0,0,0,0,0,1,0
2,2023,2.702395e+10,Race Not Available,6,105000.0,105000.0,NaN,6.125,8.92,90.13,10,0.0,0,NaN,0,0,0,0,0,0,0,1,0
3,2023,2.717110e+10,Race Not Available,6,285000.0,285000.0,NaN,6.750,13.27,104.37,10,0.0,0,NaN,0,0,0,0,0,0,0,1,0
4,2023,2.715948e+10,Race Not Available,6,175000.0,175000.0,NaN,6.875,5.87,82.22,10,0.0,0,NaN,0,0,0,0,0,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
100926,2023,2.700902e+10,Race Not Available,6,255000.0,265000.0,NaN,6.125,4.88,106.18,10,0.0,0,NaN,0,0,0,0,0,0,0,1,0
100927,2023,2.706778e+10,Race Not Available,6,185000.0,335000.0,NaN,6.125,46.58,74.68,10,0.0,0,NaN,0,0,0,0,0,0,0,1,0
100928,2023,2.714948e+10,Race Not Available,6,65000.0,75000.0,NaN,6.750,20.14,118.83,10,0.0,0,NaN,0,0,0,0,0,0,0,1,0
100929,2023,2.705913e+10,Race Not Available,6,545000.0,525000.0,NaN,5.000,6.84,90.07,10,0.0,0,NaN,0,0,0,0,0,0,0,1,0


In [14]:
X = df_mortgage.drop(['activity_year', 'census_tract', 'derived_race', 'action_taken', 'loan_amount', 'property_value',
                     'income', 'interest_rate', 'denial_reason-1', 'covenant_count', 'covenant_density'], axis = 1)
y = df_mortgage['was_approved']

In [15]:
X

,tract_minority_population_percent,tract_to_msa_income_percentage,was_approved,2 or more minority races,American Indian or Alaska Native,Asian,Black or African American,Free Form Text Only,Joint,Native Hawaiian or Other Pacific Islander,Race Not Available,White
0,40.37,73.48,0,0,0,0,0,0,0,0,1,0
1,10.64,121.26,0,0,0,0,0,0,0,0,1,0
2,8.92,90.13,0,0,0,0,0,0,0,0,1,0
3,13.27,104.37,0,0,0,0,0,0,0,0,1,0
4,5.87,82.22,0,0,0,0,0,0,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...
100926,4.88,106.18,0,0,0,0,0,0,0,0,1,0
100927,46.58,74.68,0,0,0,0,0,0,0,0,1,0
100928,20.14,118.83,0,0,0,0,0,0,0,0,1,0
100929,6.84,90.07,0,0,0,0,0,0,0,0,1,0


In [16]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.3, train_size = 0.7, random_state = 42)

In [17]:
#Checking training and test sets.
print('---------------------------------------------------------------')
print('y training set' + str(y_train.shape))
print('y test set' + str(y_test.shape))
print('Shape of y' + str(y.shape))
print('---------------------------------------------------------------')
print('***************************************************************')
print('X training set' + str(X_train.shape))
print('X test set' + str(X_test.shape))
print('Shape of X' + str(X.shape))

print('---------------------------------------------------------------')

---------------------------------------------------------------
y training set(70651,)
y test set(30280,)
Shape of y(100931,)
---------------------------------------------------------------
***************************************************************
X training set(70651, 12)
X test set(30280, 12)
Shape of X(100931, 12)
---------------------------------------------------------------


### Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression, Ridge, Lasso
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

### Ridge Regression